# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list all record sets and fields, showing their `@id` for reference in later steps.

In [ ]:
from pprint import pprint

# List all record sets and their @id
print("Available Record Sets:\n")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    - Field name: {f.name}")
        print(f"      @id: {f.id}")
        print(f"      Data type: {f.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract all record sets and load each to a pandas DataFrame for analysis. You can use the `@id` of a record set and its fields for further processing.

In [ ]:
# Collect the record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

print("Record Set @ids:")
pprint(record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nRecordSet {record_set_id} loaded: {len(df)} records.")
        print("Columns:", df.columns.tolist())
        display(df.head())
    else:
        print(f"\nRecordSet {record_set_id} is empty or could not be loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.
Below, pick one record set and relevant fields for demonstration (replace with actual field `@id` as shown in the overview for your particular analysis).

In [ ]:
# Example: Use the first record set if available for demonstration
if record_set_ids and record_set_ids[0] in dataframes:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Find a numeric field
    numeric_field_id = None
    for f in dataset.metadata.get_record_set(record_set_id).fields:
        # Try to choose a 'Float' or 'Integer' field
        if f.data_type in ['schema:Float', 'schema:Integer', 'Float', 'Integer', 'Number']:
            if f.id in df.columns:
                numeric_field_id = f.id
                break
    
    print(f"Using numeric field: {numeric_field_id}")

    # Example filtering
    threshold = df[numeric_field_id].mean() if numeric_field_id else None
    
    filtered_df = None
    if numeric_field_id and threshold is not None:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        norm_field = f"{numeric_field_id}_normalized"
        filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_field]].head())

        # Try to select a categorical/group field (e.g., anything string/object)
        group_field = None
        for f in dataset.metadata.get_record_set(record_set_id).fields:
            # Skip the numeric field, look for a likely grouping
            if f.id != numeric_field_id and f.data_type not in ['schema:Float', 'schema:Integer', 'Float', 'Integer', 'Number']:
                if f.id in df.columns and df[f.id].dtype == 'object':
                    group_field = f.id
                    break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print('No suitable grouping field found.')
    else:
        print('No numeric field found for EDA.')
else:
    print('No loaded record set available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
The following cell demonstrates basic visualization with matplotlib and seaborn. Adapt the field IDs as needed based on your data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If there is a suitable numeric field and filtered_df from EDA:
if 'filtered_df' in locals() and filtered_df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=30, color='royalblue')
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} (filtered > mean)")
    plt.show()

    # If grouped_df exists, plot bar plot
    if 'grouped_df' in locals() and group_field is not None and not grouped_df.empty:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id, palette='viridis')
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.title(f"Mean {numeric_field_id} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print('No filtered data available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded metadata and records using the Croissant schema via `mlcroissant`.
- We reviewed available record sets and their fields, referencing all entities by their `@id`.
- We demonstrated filtering, normalization, and grouping on a numeric field, with basic visualization.
- For your exact analysis, consult the above data overview for all available record and field `@id`s, and adapt fields and EDA as appropriate.